# Part d: L1 and L2 regularization


We add the hyperparameter $\lambda$ with the $L_2$- and $L_1$-norms of the weights to the 1D Runge function. 

The goal is to find the optimal penalty ($\lambda$), learning rate ($\gamma$) and architecture on validation data, and compare the test MSE to Ridge and Lasso regression from Project 1 on the same data set. 

Cost minimized by the network:

* $C = MSE + \lambda \sum_l ||W^l ||$

* $||W^l ||_1 = \sum |w|(L1)$

* $||W||^2_2 = \sum w^2 (L2)$

The biases are not penalized (penalize_bias=False), in the same way as the intercept is not penalized in Ridge and Lasso. 

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax

from src.data import make_runge_split, make_runge_split_3, design_matrix, scale_data, center_data, runge
from src.functions import mse, mse_der, sigmoid, sigmoid_der, identity, identity_der
from src.network import NeuralNetwork
from src.optimizers import Adam
from src.regression import ols_fit, ridge_fit, lasso_fit
from src.metrics import r2
from src.plotting import line_plot, heatmap, save_fig

### Recreating Ridge and Lasso results from Project 1

Dataset and best result from project 1: 

* n = 100 points, $\sigma$ = 0.1 noise, seed = 2026, test size = 0.2, optimization method = Adam

Splitting the data:

* `make_runge_split` separates data into 80/20 training- and testpoints. Ridge, Lasso and the final network are fitted to these 80 training points. 

* `make_runge_split_3` takes 16 validation points out of the 80 (64/16/20). The hyperparameters are chosen on the validation set. The test set is identical in both splits and used only once, at the end.

In [ ]:
x_train, x_test, y_train, y_test = make_runge_split()                   # split 80/20
x_trn, x_val, x_tst, y_trn, y_val, y_tst = make_runge_split_3()         # 64/16/20, for tuning
assert np.array_equal(x_tst, x_test) and np.array_equal(y_tst, y_test)  # same test set in both splits

print("train", x_train.shape, " test", x_test.shape)
print("tuning: train", x_trn.shape, " validation", x_val.shape)

degree = 8                      # best OLS and Ridge
#degree_lasso = 5               # test        
num_iters = 20000 
lam_ridge = 1e-4/len(x_train)   # best Ridge
lam_lasso = 1e-2                # used for Lasso
Gamma = 0.01
Seed = 2026

In [ ]:
# design matrix
def prepare(deg):
    matrix_train = design_matrix(x_train, deg, intercept=False)
    matrix_test = design_matrix(x_test, deg, intercept=False)
    P_train, P_test, P_mean, P_std = scale_data(matrix_train, matrix_test)
    
    def poly_pred(theta, x):
        P = (design_matrix(x, deg, intercept=False) - P_mean) / P_std
        return P @ theta + y_mean
    return P_train, P_test, poly_pred

P8_train, P8_test, predict8 = prepare(8)     # OLS and Ridge
#P5_train, P5_test, predict5 = prepare(5)    # test
y_train_c, y_test_c, y_mean = center_data(y_train, y_test)

# regression fits
theta_ols = ols_fit(P8_train, y_train_c)
theta_ridge = ridge_fit(P8_train, y_train_c, lam_ridge)
theta_lasso = lasso_fit(P8_train, y_train_c, lam_lasso, gamma = Gamma, n_iter = num_iters)

# predictions
results = {}
for name, theta, predict in [(f'OLS   - deg = {degree}', theta_ols, predict8),
                    (f'Ridge - deg = {degree}, lam = {lam_ridge}', theta_ridge, predict8),
                    (f'Lasso - deg = {degree}, lam = {lam_lasso}', theta_lasso, predict8)]:
    
    y_pred = predict(theta, x_test)
    MSE = mse(y_pred, y_test)
    R2 = r2(y_pred, y_test)
    results[name] = (float(MSE), R2)
    print(f'{name:32s} | test MSE {results[name][0]:.4f} | R2 {results[name][1]:.3f}')

### Using 1D Runge function
#### Adding hyperparameters $\lambda$ with the $L_2$ and $L_1$ norms (sec 8.12).

* `train_net` builds a new network and a new `Adam` for every run. Adam stores its moving averages, so reusing one optimizer would carry the state of the previous run
into the next. Every run uses the same seed for the initial weights and for the minibatches, so differences
between runs come from the hyperparameters alone.

In [ ]:
X_trn_s, X_val_s, x_mean_trn, x_std_trn = scale_data(x_trn, x_val) # scale with 64 tuning-training points
epochs = 500
batch_size = 14

def make_net(lam, reg, gamma=Gamma, hidden=[50]):
    # input - hidden layers - linear output, MSE cost, Adam, (not penalising biases)
    k = len(hidden)
    return NeuralNetwork(network_input_size = 1,
                        layer_output_sizes = hidden + [1],
                        activation_funcs = [sigmoid] * k +  [identity], 
                        activation_ders = [sigmoid_der] * k + [identity_der],
                        cost_fun = mse,
                        cost_der = mse_der,
                        optimizer = Adam(gamma),
                        regularization = reg,
                        lmbda = lam,
                        seed = Seed,
                        penalize_bias=False)


def train_net(X_fit, y_fit, X_eval, y_eval, reg, lam, gamma = Gamma, hidden = [50]):
    # training the network on the fits and returning it + MSE on the validations
    neural_net = make_net(lam, reg, gamma, hidden)
    neural_net.train(X_fit, y_fit, epochs=epochs, batch_size=batch_size, rng = np.random.default_rng(Seed))
    MSE = mse(neural_net.predict(X_eval), y_eval)
    score = float(MSE)
    return neural_net, (score if np.isfinite(score) else np.nan)


def gradient_check(network, X, y): 
    # compare compute_gradient with jax.grad
    def cost_layers(layers, X, y):
        network.layers = layers
        return network.cost(X, y, include_penalty=True)
    
    net_layers = network.layers
    grad = network.compute_gradient(X, y)
    grad_jax = jax.grad(cost_layers)(net_layers, X, y) 
    network.layers = net_layers # put the real weights back
    return [(bool(np.allclose(dW, dW_jax)), bool(np.allclose(db, db_jax))) for (dW, db), (dW_jax, db_jax) in zip(grad, grad_jax)]



In [ ]:
# Adding L2-Ridge to network
L2_net = make_net(reg = 'L2', lam = 1e-2) # lam_ridge too small for testing the penalty
print('L2 gradient check against jax, (dW, db) per layer:', gradient_check(L2_net, X_trn_s, y_trn))

# lambda scan
for l in [0.0, 1e-4, 1e-2]:
    val = train_net(X_trn_s, y_trn, X_val_s, y_val, reg = 'L2', lam = l)[1]
    print(f'L2: lam = {l:6g}: validation MSE {val:.4f}')

In [ ]:
# Adding L1-Lasso to network
L1_net = make_net(lam = lam_lasso, reg = 'L1')
print('L1 gradient check against jax, (dW, db) per layer:', gradient_check(L1_net, X_trn_s, y_trn))

# lambda scan
for l in [0.0, 1e-4, 1e-2]:
    val = train_net(X_trn_s, y_trn, X_val_s, y_val, reg = 'L1', lam = l)[1]
    print(f'L1: lam = {l:6g}: validation MSE {val:.4f}')

### Finding the best result for the hyperparameters ($\lambda$) and learning rate ($\gamma$)

We start from a well-performing set, then change two hyperparameters at a time:

1. $\lambda \times \gamma$ for one hidden layer with 50 nodes (one of the architectures from part b).
2. $\lambda \times$ architecture at the best $\gamma$ from step 1.

Every cell is the validation MSE. Diverged runs (MSE>1) are left blank. $\lambda = 0$ is the unregularised network and it is identical in the L1- and L2-maps. 

In [ ]:
# parameter scans
lam_vals = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
gamma_vals = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]

val_mse, best = {}, {}
for reg in ['L2', 'L1']:
    Z = np.full((len(lam_vals), len(gamma_vals)), np.nan)
    for i, lam in enumerate(lam_vals):
        for j, gam in enumerate(gamma_vals):
            Z[i,j] = train_net(X_trn_s, y_trn, X_val_s, y_val, reg, lam, gam)[1]
    
    Z[Z > 1] = np.nan # removing diverged runs
    val_mse[reg] = Z
    i, j = np.unravel_index(np.nanargmin(Z), Z.shape)
    best[reg] = dict(hidden = [50], lam = lam_vals[i], gam = gamma_vals[j], val = Z[i, j])

    heatmap(Z, gamma_vals, lam_vals, r'learning rate $\gamma$', r'$\lambda$ ('+ reg + ')', label = 'validation MSE', name = f'd_heatmap_{reg}')
    plt.show()
    print(f'{reg}: best lambda = {lam_vals[i]:g}, gamma = {gamma_vals[j]:g}, validation MSE {Z[i, j]:.4f}')


In [ ]:
# architecture scan
architectures = [[10], [50], [100], [50, 50], [100, 100]]
arch_labels = ['-'.join(map(str, a)) for a in architectures]

for reg in ['L2', 'L1']:
    gam_ = best[reg]['gam']
    Z = np.full((len(lam_vals), len(architectures)), np.nan)
    for i, lam in enumerate(lam_vals):
        for j, hidden in enumerate(architectures):
            Z[i,j] = train_net(X_trn_s, y_trn, X_val_s, y_val, reg, lam, gam_, hidden)[1]
    
    Z[Z > 1] = np.nan # removing diverged runs
    i, j = np.unravel_index(np.nanargmin(Z), Z.shape)
    best[reg] = dict(hidden = architectures[j], lam = lam_vals[i], gam = gam_, val = Z[i, j])

    fig, ax = heatmap(Z, range(len(architectures)), lam_vals, 'hidden layers (nodes per layer)', r'$\lambda$ ('+ reg + ')', label = 'validation MSE')
    ax.set_xticklabels(arch_labels)
    save_fig(f'd_heatmap_arch_{reg}')
    plt.show()
    print(f'{reg}: best architecture = {arch_labels[j]}, lambda = {lam_vals[i]:g}, gamma = {gam_:g}, validation MSE {Z[i, j]:.4f}')


### Comparing the final models to the Project 1 results

The networks are retrained on all 80 training points (same as Ridge and Lasso) and evaluated once on the test set. We also train the network on $\lambda = 0$ to see what the penalty itself adds. 

In [ ]:
X_train_s, X_test_s, x_mean, x_std = scale_data(x_train, x_test)
x_plot = np.linspace(-1, 1, 300).reshape(-1, 1)

final_nets = {}
for reg in ['L2','L1']:
    config = best[reg]
    for lam, label in [(config['lam'], f"FFNN {'-'.join(map(str, config['hidden']))} + {reg}, lambda {config['lam']:g}"),
                         (0.0, f"FFNN {'-'.join(map(str, config['hidden']))}, no penalty ({reg} settings)")]:
        neural_net = train_net(X_train_s, y_train, X_test_s, y_test, reg, lam, config['gam'], config['hidden'])[0]
        y_pred = neural_net.predict(X_test_s)
        MSE = mse(y_pred, y_test)
        R2 = r2(y_pred, y_test)
        results[label] = (float(MSE), R2)
        final_nets[(reg, lam)] = neural_net     

In [ ]:
# Compare the $L_2$ results with Ridge regression from project 1
L2_config = best['L2']
L2_network = final_nets[('L2', L2_config['lam'])]
for name in [k for k in results if k.startswith(('Ridge')) or 'L2' in k]:
    print(f'{name:50s} test MSE {results[name][0]:.4f}  R2 {results[name][1]:.4f}')

line_plot(x_plot,
         {'Runge function': runge(x_plot),
         f'Ridge, degree {degree}': predict8(theta_ridge, x_plot),
         f"FFNN + L2 $\\lambda$ = {L2_config['lam']:g}": L2_network.predict((x_plot - x_mean) / x_std)},
         '$x$', '$y$', data=(x_train, y_train), name = 'd_fit_L2_ridge')
plt.show()

In [ ]:
# Compare the $L_1$ results with the Lasso calculations of project 1
L1_config = best['L1']
L1_network = final_nets[('L1', L1_config['lam'])]
for name in [k for k in results if k.startswith('Lasso') or 'L1' in k]:
    print(f'{name:50s} test MSE {results[name][0]:.4f}  R2 {results[name][1]:.4f}')

line_plot(x_plot,
         {'Runge function': runge(x_plot),
         f'Lasso, degree {degree}': predict8(theta_lasso, x_plot),
         f"FFNN + L1 $\\lambda$ = {L1_config['lam']:g}": L1_network.predict((x_plot - x_mean) / x_std)},
         '$x$', '$y$', data=(x_train, y_train), name = 'd_fit_L1_lasso')
plt.show()

### Does L1 set weights to zero?

Lasso sets some coefficients to exactly 0, while our method only push them towards 0. 

Therefore, we count weights with $|w| < 10^{-3}$, and compare the
distribution of the weights of the L1 network, the L2 network and the unpenalised network.

In [ ]:
def all_weights(net):
    # all weights of network in one flat array (ex biases, not penalized)
    return np.concatenate([np.ravel(W) for W,_ in net.layers])
    
print('Lasso, our code (|theta| < 1e-3):', np.sum(np.abs(theta_lasso) < 1e-3), 'of', theta_lasso.size)

nets_W = {f"L1 $\\lambda$ = {best['L1']['lam']:g}": final_nets[('L1', best['L1']['lam'])],
          f"L2 $\\lambda$ = {best['L2']['lam']:g}": final_nets[('L2', best['L2']['lam'])],
          'no penalty': final_nets[('L1', 0.0)]}


fig, ax = plt.subplots()
bins = np.linspace(-8, 1, 46)           # bins in log10|w|

for label, net in nets_W.items():
    w = all_weights(net)
    print(f'{label:24s}: {np.mean(np.abs(w) < 1e-3):.2%} of {w.size} weights with |w| < 1e-3')
    ax.hist(np.log10(np.abs(w) + 1e-12), bins = bins, histtype = 'step', density=True, label = label)
ax.set_xlabel(r"$\log_{10}|w|$")
ax.set_ylabel("Fraction of weights (density)")  
ax.legend()
save_fig("d_weight_hist")
plt.show()

### Summary of test results

The hyperparameters of the networks were chosen on the validation set. Every model below was fitted on the same 80 training points and evaluated once on the same 20 test points. Differences of a few times $10^{-3}$ in the MSE are within the noise (due to small test size).

In [ ]:
print(f"{'model':43s} {'test MSE':5s} {'R2':>8s}")
for name, (test_mse, test_r2) in sorted(results.items(), key=lambda kv: kv[1][0]):
    print(f'{name:40s} {test_mse:10.4f} {test_r2:8.3f}')

print("\nchosen hyperparameters:")
for reg, config in best.items():
    print(f"{reg}: hidden {config['hidden']}, lambda {config['lam']:g}, gamma {config['gam']:g}, (validation MSE {config['val']:.4f})")